# Training Rectified Flow for Data-to-Data Interpolation

This notebook implements a **training pipeline** for Rectified Flow on FashionMNIST data-to-data tasks

## Core Concept

A Rectified Flow learns a neural ODE velocity field that transports a source distribution to a target distribution. We optimize:

$$\mathcal{L} = \mathbb{E}_{t \sim U[0,1], x_a, x_b} \left[ \left\| v_\theta(z_t, t) - (x_b - x_a) \right\|^2 \right]$$

where:
- $z_t = (1-t) x_a + t x_b$ (straight-line path from source to target)
- $v_\theta$ is the learned velocity field (neural network)
- Target velocity is $(x_b - x_a)$

## Key Properties

1. **Straight-line interpolation**: Data points move linearly from source to target
2. **Velocity matching**: Network learns to predict exact endpoint transformation
3. **ODE sampling**: At test time, integrate velocity forward to generate trajectories


## Prerequisite: Colab Setup

Run all cells in [`00_colab_setup.ipynb`](00_colab_setup.ipynb) first. Notebook 00 owns repository cloning, working-directory setup, and the Python import path. This notebook assumes that setup is complete.

In [ ]:
# check environment
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

In [ ]:
# core imports
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
from dataclasses import dataclass, field
from typing import Dict, Optional, List

# use project APIs
from api.flow_model_api import (
    DatasetSpec,
    ModelSpec,
    DataToDataRF,
    build_dataset_loaders,
    build_flow_model,
    save_checkpoint,
    load_checkpoint,
)

print('Imports successful')

In [ ]:
# disable Weights & Biases (for clean training without external logging)
os.environ['WANDB_DISABLED'] = 'true'
print('W&B disabled')

## Training Configuration


### FashionMNIST Dataset Overview

FashionMNIST contains 10 classes of clothing items. Default configuration trains on:

| Class ID | Name | Role in Training |
|----------|------|------------------|
| 0 | T-shirt/top | - |
| 1 | Trouser | - |
| 2 | Pullover | - |
| 3 | Dress | - |
| 4 | Coat | - |
| 5 | Sandal | - |
| 6 | Shirt | - |
| **7** | **Sneaker** | **Source (x_a)** |
| 8 | Bag | - |
| **9** | **Ankle Boot** | **Target (x_b)** |

**Interpolation Task**: Model learns to transport images from Sneaker class to Ankle Boot class.

This pairing was chosen because both are footwear with distinct visual features, making it a meaningful semantic transformation task


In [ ]:
@dataclass
class TrainingConfig:
    """Encapsulates all training hyperparameters"""
    # dataset configuration
    dataset_name: str = 'fashionmnist'
    source_class: int = 7  # Sneaker
    target_class: int = 9  # Ankle Boot
    root: str = 'data'
    
    # model configuration
    model_profile: str = 'medium'  # small, medium, large
    
    # training hyperparameters
    batch_size: int = 128
    test_batch_size: int = 128
    epochs: int = 100
    lr: float = 2e-4
    
    # early stopping
    early_stopping_patience: int = 10
    early_stopping_min_delta: float = 1e-4
    
    # checkpointing
    checkpoint_dir: str = 'assets/fmnist_d2d_runs'
    model_name: str = 'fmnist_d2d_training'
    
    # sampling during training
    sample_every_n_epochs: int = 10
    sample_steps: int = 50
    sample_count: int = 5  # number of trajectories to sample
    output_dir: str = 'contents'

# create configuration instance
config = TrainingConfig()
print('Configuration created')
print(f'  Source class: {config.source_class} (Sneaker)')
print(f'  Target class: {config.target_class} (Ankle Boot)')
print(f'  Model: {config.model_profile}')
print(f'  Epochs: {config.epochs}')

## Data Loading

In [ ]:
# build dataset specification
dataset_spec = DatasetSpec(
    name=config.dataset_name,
    root=config.root,
    source_class=config.source_class,
    target_class=config.target_class,
    batch_size=config.batch_size,
    test_batch_size=config.test_batch_size,
)

# load datasets and dataloaders
loaders_dict = build_dataset_loaders(
    dataset_spec,
    batch_size=config.batch_size,
    test_batch_size=config.test_batch_size,
    download=True,
)

train_loader = loaders_dict['train_loader']
test_loader = loaders_dict['test_loader']
train_dataset = loaders_dict['train_dataset']

print(f'Train loader: {len(train_loader)} batches')
print(f'Test loader: {len(test_loader)} batches')
print(f'Total train samples: {len(train_dataset)}')

## Model Initialization

In [ ]:
# build model specification
model_spec = ModelSpec(
    model_profile=config.model_profile,
    channels=1,
    image_size=32,
    num_classes=1,
    lr=config.lr,
    checkpoint_dir=config.checkpoint_dir,
)

# build model, flow wrapper, and optimizer
model, flow_model, optimizer = build_flow_model(model_spec)
model = model.to(device)

# count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model profile: {config.model_profile}')

## Training Loop

For each epoch:
1. **Forward pass**: $z_t = (1-t) x_a + t x_b$, predict $v_\theta(z_t, t)$
2. **Loss**: MSE between predicted and target velocity
3. **Backward pass**: Update model parameters
4. **Validation**: Evaluate on test set, track best model
5. **Early stopping**: Stop if validation loss plateaus


In [ ]:
def train_epoch(model, flow_model, train_loader, optimizer, device):
    """Train for one epoch. Return average loss"""
    model.train()
    total_loss = 0.0
    count = 0
    
    for x_a, x_b in train_loader:
        x_a = x_a.to(device)
        x_b = x_b.to(device)
        
        # forward pass: compute loss
        loss = flow_model.forward(x_a, x_b)
        
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        count += 1
    
    return total_loss / max(1, count)

def evaluate_epoch(model, flow_model, test_loader, device):
    """Evaluate on test set without gradients. Return average loss"""
    model.eval()
    total_loss = 0.0
    count = 0
    
    with torch.no_grad():
        for x_a, x_b in test_loader:
            x_a = x_a.to(device)
            x_b = x_b.to(device)
            loss = flow_model.forward(x_a, x_b)
            total_loss += loss.item()
            count += 1
    
    return total_loss / max(1, count)

print('Training functions defined')

In [ ]:
# prepare checkpoint directory
checkpoint_dir = Path(config.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_latest = checkpoint_dir / f'{config.model_name}_checkpoint_latest.pt'
checkpoint_best = checkpoint_dir / f'{config.model_name}_checkpoint_best.pt'

print(f'Checkpoints will be saved to: {checkpoint_dir}')

In [ ]:
# training loop with early stopping
history = {
    'train_loss': [],
    'test_loss': [],
}

best_test_loss = float('inf')
patience_counter = 0
epoch_with_best = 0

print(f'Starting training for {config.epochs} epochs...\n')

for epoch in range(config.epochs):
    # train
    train_loss = train_epoch(model, flow_model, train_loader, optimizer, device)
    
    # evaluate
    test_loss = evaluate_epoch(model, flow_model, test_loader, device)
    
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    
    # save latest checkpoint (with minimal config)
    save_checkpoint(model, optimizer, checkpoint_latest, config={})
    
    # track best model
    if test_loss < best_test_loss - config.early_stopping_min_delta:
        best_test_loss = test_loss
        patience_counter = 0
        epoch_with_best = epoch
        save_checkpoint(model, optimizer, checkpoint_best, config={})
    else:
        patience_counter += 1
    
    # print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | Train loss: {train_loss:.6f} | Test loss: {test_loss:.6f} | '
              f'Best: {best_test_loss:.6f} @ {epoch_with_best+1} | Patience: {patience_counter}/{config.early_stopping_patience}')
    
    # early stopping
    if patience_counter >= config.early_stopping_patience:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nTraining completed. Best test loss: {best_test_loss:.6f} at epoch {epoch_with_best+1}')

## Training Summary

In [ ]:
import matplotlib.pyplot as plt

# plot training curves
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history['test_loss'], label='Test Loss', linewidth=2)
plt.axvline(x=epoch_with_best, color='red', linestyle='--', label=f'Best epoch ({epoch_with_best+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Rectified Flow Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150)
plt.show()

print(f'Training curve saved')

In [ ]:
# save training history
history_file = output_dir / 'training_history.json'
with open(history_file, 'w') as f:
    json.dump(history, f, indent=2)

print(f'History saved to {history_file}')

In [ ]:
# print final statistics
print('\n' + '='*60)
print('TRAINING COMPLETED')
print('='*60)
print(f'Total epochs trained: {len(history["train_loss"])}')
print(f'Best epoch: {epoch_with_best + 1}')
print(f'Best test loss: {best_test_loss:.6f}')
print(f'Final train loss: {history["train_loss"][-1]:.6f}')
print(f'Final test loss: {history["test_loss"][-1]:.6f}')
print(f'\nCheckpoint saved: {checkpoint_best}')
print('='*60)

## Load Best Model for Evaluation


In [ ]:
# load best checkpoint for subsequent evaluation
# recreate model and optimizer
best_model, _, best_optimizer = build_flow_model(model_spec)
best_model = best_model.to(device)

# load checkpoint state
checkpoint = load_checkpoint(
    best_model,
    best_optimizer,
    checkpoint_best,
    device=device
)

best_flow_model = DataToDataRF(best_model)

print(f'Best model loaded from epoch {epoch_with_best + 1}')
print(f'Test loss at that epoch: {best_test_loss:.6f}')